# Graph-of-Thought — Colab GPU runner (T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arrafmousa/graph-of-thought/blob/master/notebooks/colab_run.ipynb)

Runs the **reasoning-graph POC** on GSM8K with a **frozen** `meta-llama/Llama-3.2-1B-Instruct`
(FP16), following `AGENTS.md` §31. Set **Runtime → Change runtime type → T4 GPU** first.

Two phases:
1. **Tuning** (`scripts/tune_graph.py`) — sweep merge heuristics × thresholds over 25
   questions; emits one dashboard per configuration + a comparison dashboard.
2. **Full run** (`scripts/generate_graphs.py`) — generate 100 graphs with the configured
   candidate (`hidden_cosine @ 0.95`) and render a sampled per-question graph report.

`Llama-3.2` is **gated**: request access on its model page and add a Hugging Face token
as a Colab secret named `HF_TOKEN`. Dataset + model are chosen in the config files
(explicit HF ids/revisions). Each phase writes `output/<run_id>/`; the last cell validates
both runs and downloads one ZIP containing their complete artifacts plus an offline index.

In [ ]:
# 1) Clone the repo
# Public repo:
!git clone https://github.com/arrafmousa/graph-of-thought.git

# Private repo instead? Store a GitHub token as a Colab secret named GH_TOKEN, then:
# from google.colab import userdata
# tok = userdata.get('GH_TOKEN')
# !git clone https://{tok}@github.com/arrafmousa/graph-of-thought.git

%cd graph-of-thought

In [ ]:
# 2) Install training deps (Colab already ships a CUDA build of torch)
!pip install -q -r requirements.txt
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 3) Authenticate for the gated Llama-3.2 model (needs an HF_TOKEN Colab secret)
from google.colab import userdata
from huggingface_hub import login

login(userdata.get('HF_TOKEN'))  # Colab secret; never committed (AGENTS.md section 8)


## Phase 1 — hyperparameter tuning (merge heuristics × thresholds)

Sweeps every merge heuristic and threshold over 25 GSM8K questions and emits **one
dashboard per configuration** plus a comparison dashboard and `tuning_summary.json`.
Inspect the merge samples (highest-similarity vs borderline) to pick the heuristic +
threshold that merge semantically-equivalent states without collapsing incompatible ones.


In [ ]:
# Phase 1: run the tuning sweep, then preview the comparison dashboard inline
import glob, os, json
from IPython.display import HTML, display

get_ipython().system('python scripts/tune_graph.py --config configs/tuning_pipeline/gsm8k/llama1b_gsm8k_merge_sweep.json')

latest = sorted(glob.glob('output/*tune-gsm8k*/'))[-1].rstrip('/')
manifest = json.load(open(os.path.join(latest, 'run_manifest.json')))
print('Tuning run:', os.path.basename(latest), '| status:', manifest['status'])
if manifest['status'] != 'completed':
    # Interrupted (Ctrl+C) or failed runs have no outputs yet.
    print('Run did not complete. Errors:', manifest.get('errors'))
else:
    print('configs evaluated:', manifest['outputs']['configs_evaluated'])
    display(HTML(open(os.path.join(latest, manifest['outputs']['comparison_dashboard'])).read()))


## Phase 2 — full run with the selected heuristic + threshold

Set `graph.heuristic` and `graph.threshold` in `configs/graph_pipeline/gsm8k/llama1b_gsm8k_graphs.json`
to the choice from Phase 1, then run the full graph generation below.


In [ ]:
# Phase 2: generate reasoning graphs on GSM8K with the selected heuristic/threshold
!python scripts/generate_graphs.py --config configs/graph_pipeline/gsm8k/llama1b_gsm8k_graphs.json


In [ ]:
# 5) Validate, bundle, and download both complete runs (Colab disk is ephemeral)
from pathlib import Path
import html
import json
import subprocess
import sys
import zipfile

from google.colab import files
from IPython.display import HTML, display


def latest_run(pattern):
    matches = list(Path("output").glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No run matched output/{pattern}")
    return max(matches, key=lambda path: path.stat().st_mtime)


tuning_run = latest_run("*tune-gsm8k*")
graph_run = latest_run("*graph-gsm8k*")
runs = (tuning_run, graph_run)
manifests = {}

for run_dir in runs:
    manifest = json.loads((run_dir / "run_manifest.json").read_text(encoding="utf-8"))
    if manifest["status"] != "completed":
        raise RuntimeError(
            f"{run_dir.name} did not complete: {manifest.get('errors', [])}"
        )
    subprocess.run(
        [sys.executable, "scripts/validate_run.py", str(run_dir)],
        check=True,
    )
    manifests[run_dir.name] = manifest

tuning_manifest = manifests[tuning_run.name]
graph_manifest = manifests[graph_run.name]
comparison_path = tuning_manifest["outputs"]["comparison_dashboard"]
tuning_summary_path = tuning_manifest["outputs"]["tuning_summary"]
graph_report_path = graph_manifest["outputs"]["reports"][0]

bundle_name = f"reasoning_graph_results_{graph_run.name}"
archive_path = Path(f"{bundle_name}.zip")
index_html = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Reasoning graph results</title>
  <style>
    body {{ font: 16px/1.5 sans-serif; max-width: 840px; margin: 40px auto; padding: 0 20px; }}
    li {{ margin: 10px 0; }}
    code {{ background: #f2f2f2; padding: 2px 5px; }}
  </style>
</head>
<body>
  <h1>Reasoning graph results</h1>
  <p>This bundle contains the complete tuning and graph-generation runs, including
  manifests, telemetry, raw token traces, hidden states, graph JSON, and static reports.</p>
  <h2>Start here</h2>
  <ul>
    <li><a href="output/{html.escape(tuning_run.name)}/{html.escape(comparison_path)}">Tuning comparison dashboard</a></li>
    <li><a href="output/{html.escape(tuning_run.name)}/{html.escape(tuning_summary_path)}">Machine-readable tuning summary</a></li>
    <li><a href="output/{html.escape(graph_run.name)}/dashboard.html">Graph run dashboard</a></li>
    <li><a href="output/{html.escape(graph_run.name)}/{html.escape(graph_report_path)}">Sample reasoning graph report</a></li>
  </ul>
  <p>All pages are static and can be opened without a server.</p>
</body>
</html>
"""
readme = f"""Reasoning graph result bundle

Tuning run: {tuning_run.name}
Graph run:  {graph_run.name}

Open index.html first. To use the repository validators, copy both directories from
this bundle's output/ directory into the repository's output/ directory, then run:

python scripts/validate_run.py output/{tuning_run.name}
python scripts/validate_run.py output/{graph_run.name}
"""

with zipfile.ZipFile(
    archive_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    archive.writestr(f"{bundle_name}/index.html", index_html)
    archive.writestr(f"{bundle_name}/README.txt", readme)
    for run_dir in runs:
        for artifact in run_dir.rglob("*"):
            if artifact.is_file():
                archive.write(
                    artifact,
                    arcname=f"{bundle_name}/{artifact.as_posix()}",
                )

size_gib = archive_path.stat().st_size / (1024 ** 3)
print(f"Bundle: {archive_path} ({size_gib:.2f} GiB)")
print(f"Tuning run: {tuning_run.name}")
print(f"Graph run: {graph_run.name}")
files.download(str(archive_path))

display(HTML((graph_run / graph_report_path).read_text(encoding="utf-8")))

### Optional
- **CPU smoke test (no GPU, no token):** tuning `!python scripts/tune_graph.py --config configs/tuning_pipeline/demo/synthetic_cpu_merge_sweep.json` then full run `!python scripts/generate_graphs.py --config configs/graph_pipeline/demo/synthetic_cpu_graphs.json`
- **Validate a downloaded run locally:** `python scripts/validate_run.py output/<run_id>`
- **Sentiment fine-tuning demo (separate workload):** `!python scripts/train.py --config configs/train_pipeline/sst2/distilbert_sst2_finetune.json`
